# Embedding Analysis

This notebook inspects the fused multimodal features used by the engagement classifier. It applies PCA to the fused vectors and colors points by engagement label to see whether High, Medium, and Low examples separate in feature space.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from config import SAMPLE_DATA_PATH, FEATURE_CACHE_PATH
from feature_extraction import MultimodalFeatureExtractor
from utils import load_sample_data, ensure_directories

ensure_directories()
df = load_sample_data(SAMPLE_DATA_PATH)
df.head()

In [ ]:
if FEATURE_CACHE_PATH.exists():
    X = np.load(FEATURE_CACHE_PATH)
else:
    extractor = MultimodalFeatureExtractor()
    X, _ = extractor.extract_dataframe_features(df)
    np.save(FEATURE_CACHE_PATH, X)

X.shape

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)

plot_df = df.copy()
plot_df['pc1'] = coords[:, 0]
plot_df['pc2'] = coords[:, 1]

colors = {'High': '#E4572E', 'Medium': '#4C78A8', 'Low': '#5C677D'}

fig, ax = plt.subplots(figsize=(8, 6))
for label, group in plot_df.groupby('engagement_label'):
    ax.scatter(
        group['pc1'],
        group['pc2'],
        label=label,
        s=80,
        alpha=0.85,
        color=colors.get(label, '#999999'),
        edgecolor='white',
        linewidth=0.8,
    )

ax.set_title('PCA of Fused Multimodal Features')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.legend(title='Engagement')
ax.grid(alpha=0.25)
plt.show()

pca.explained_variance_ratio_

## How To Read This

If High, Medium, and Low samples form visible groups, the fused features are carrying engagement-relevant signal. If the classes overlap heavily, the dataset is too small, the labels are noisy, or the feature design needs improvement. With this synthetic sample data, treat the plot as a pipeline sanity check rather than model evidence.